```
  __  __ ___ _   _ _____  ____ ____      _    _____ _____ 
 |  \/  |_ _| \ | | ____|/ ___|  _ \    / \  |  ___|_   _|
 | |\/| || ||  \| |  _| | |   | |_) |  / _ \ | |_    | |  
 | |  | || || |\  | |___| |___| _ <  / ___ \|  _|   | |  
 |_|  |_|___|_| \_|_____|\____|_| \_\/_/   \_\_|     |_|  
```

# 🎮 Minecraft Server on Google Colab
### Made by **FTGAMERV2** | Version 2.0 | Minecraft **1.21.4**

---

**Features:**
- ✅ Minecraft **1.21.4** (latest stable)
- ✅ **Java 21** — required for 1.21.x, major performance boost
- ✅ **PaperMC** — auto fetches latest build
- ✅ **Optimized JVM flags** — Aikar's flags tuned for Java 21
- ✅ **playit.gg** tunnel — no token needed, permanent IP option
- ✅ Auto plugin folder + playit plugin setup
- ✅ Clean code, proper error handling

---
> 📌 **First time?** Run **Step 1** once to set up. After that, only run **Step 2** every time.

---
# 📦 STEP 1 — Create & Setup Server
> ⚠️ **Run this ONLY the first time.** Skip to Step 2 if you already set up the server.

In [ ]:
# ============================================================
#  CONFIGURATION — Edit these values before running
# ============================================================

# Minecraft version (1.21.x recommended)
# Tested & working: 1.21.4, 1.21.3, 1.21.1, 1.21
VERSION = '1.21.4'

# Server type:
#   'paper'  — Best performance, supports plugins (recommended)
#   'fabric' — Use for Fabric mods
#   'forge'  — Use for Forge mods (slow install ~15 min)
SERVER_TYPE = 'paper'

# Google Drive folder name for your server
SERVER_FOLDER = 'Minecraft-server'

# ============================================================

import os, json, requests
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

SERVER_PATH = f"/content/drive/My Drive/{SERVER_FOLDER}"
os.makedirs(SERVER_PATH, exist_ok=True)
%cd "{SERVER_PATH}"

print(f"✅ Server folder ready: {SERVER_PATH}")
print(f"⚙️  Version: {VERSION} | Type: {SERVER_TYPE}")

# ── Build download URLs ──────────────────────────────────────
def get_paper_url(version):
    """Fetch the latest PaperMC build URL for a given version."""
    builds_url = f"https://api.papermc.io/v2/projects/paper/versions/{version}/builds"
    resp = requests.get(builds_url, timeout=15)
    resp.raise_for_status()
    builds = resp.json()["builds"]
    latest = sorted(builds, key=lambda b: b["build"])[-1]
    build_num = latest["build"]
    jar = latest["downloads"]["application"]["name"]
    url = f"https://api.papermc.io/v2/projects/paper/versions/{version}/builds/{build_num}/downloads/{jar}"
    print(f"📥 PaperMC build #{build_num} → {jar}")
    return url

FABRIC_INSTALLER_VERSION = '1.0.1'

jar_name = {
    'paper':  'server.jar',
    'fabric': 'fabric-installer.jar',
    'forge':  'forge-installer.jar'
}

download_url = {
    'paper':  get_paper_url(VERSION) if SERVER_TYPE == 'paper' else None,
    'fabric': f'https://maven.fabricmc.net/net/fabricmc/fabric-installer/{FABRIC_INSTALLER_VERSION}/fabric-installer-{FABRIC_INSTALLER_VERSION}.jar',
    'forge':  f'https://serverjars.com/api/fetchJar/modded/forge/{VERSION}'
}

# ── Download server JAR ──────────────────────────────────────
print(f"\n🚀 Downloading {SERVER_TYPE} server...")
r = requests.get(download_url[SERVER_TYPE], timeout=60, stream=True)

if r.status_code == 200:
    jar_path = f"{SERVER_PATH}/{jar_name[SERVER_TYPE]}"
    with open(jar_path, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f"✅ Downloaded: {jar_name[SERVER_TYPE]}")
else:
    print(f"❌ Download failed! Status: {r.status_code}")
    print("   → Check if the version exists on papermc.io and try again.")
    raise SystemExit(1)

# ── Fabric/Forge extra install steps ────────────────────────
if SERVER_TYPE == 'fabric':
    print("\n⚙️  Running Fabric installer...")
    !java -jar fabric-installer.jar server -mcversion $VERSION -downloadMinecraft

if SERVER_TYPE == 'forge':
    print("\n⚙️  Running Forge installer (this takes ~15 min, please wait)...")
    !java -jar forge-installer.jar --installServer

# ── Accept EULA ──────────────────────────────────────────────
with open('eula.txt', 'w') as f:
    f.write('# By setting eula=true you agree to Mojang\'s EULA\n')
    f.write('# https://aka.ms/MinecraftEULA\n')
    f.write('eula=true\n')
print("✅ EULA accepted.")

# ── Create plugins folder ─────────────────────────────────────
PLUGINS_PATH = f"{SERVER_PATH}/plugins"
os.makedirs(PLUGINS_PATH, exist_ok=True)
print(f"✅ Plugins folder ready: {PLUGINS_PATH}")

# ── Download playit.gg Minecraft plugin ───────────────────────
print("\n🔌 Downloading playit.gg Minecraft plugin...")
try:
    playit_plugin_url = "https://github.com/playit-cloud/playit-minecraft-plugin/releases/latest/download/playit-minecraft-plugin.jar"
    pr = requests.get(playit_plugin_url, timeout=30, allow_redirects=True)
    if pr.status_code == 200:
        plugin_path = f"{PLUGINS_PATH}/playit-minecraft-plugin.jar"
        with open(plugin_path, 'wb') as f:
            f.write(pr.content)
        print(f"✅ playit plugin saved → plugins/playit-minecraft-plugin.jar")
    else:
        print(f"⚠️  playit plugin download returned {pr.status_code} — skip, CLI tunnel will still work.")
except Exception as e:
    print(f"⚠️  Could not download playit plugin: {e} — CLI tunnel will still work.")

# ── Save config ───────────────────────────────────────────────
config = {"server_type": SERVER_TYPE, "server_version": VERSION}
json.dump(config, open('colabconfig.json', 'w'))
print("✅ Config saved.")

print("\n🎉 Setup complete! Now run Step 2 to start your server.")

---
# 🚀 STEP 2 — Start the Server
> ▶️ **Run this every time you want to start the server.**

**Before running:**
- The **IP address changes every session** — check the output below after starting.
- Uses **playit.gg** tunnel — no token or account needed!
- Keep this tab **open and active** to prevent Colab from disconnecting.

In [ ]:
# ============================================================
#  CONFIGURATION
# ============================================================

# RAM allocation (Colab gives ~12 GB total; 10G for server is ideal)
MEMORY = '10G'

# Google Drive folder name (must match what you used in Step 1)
SERVER_FOLDER = 'Minecraft-server'

# ============================================================

import os, json, subprocess, threading, time, sys
from google.colab import drive

# ── Mount Drive & cd into server folder ─────────────────────
drive.mount('/content/drive')
SERVER_PATH = f"/content/drive/My Drive/{SERVER_FOLDER}"
%cd "{SERVER_PATH}"

# ── Load config ───────────────────────────────────────────────
if os.path.isfile('colabconfig.json'):
    config = json.load(open('colabconfig.json'))
else:
    config = {'server_type': 'paper', 'server_version': '1.21.4'}
    json.dump(config, open('colabconfig.json', 'w'))
    print("⚠️  No config found, using defaults (paper 1.21.4).")

server_type    = config['server_type']
server_version = config.get('server_version', '1.21.4')
print(f"📋 Loaded config: {server_type} {server_version}")

# ── Update package list (silent) ────────────────────────────
print("\n🔄 Updating package list...")
!sudo apt-get update -qq

# ── Install Java 21 (required for Minecraft 1.21.x) ─────────
print("\n☕ Installing Java 21...")
!sudo apt-get install -y -qq openjdk-21-jre-headless 2>/dev/null

# Force Java 21 as default
!sudo update-alternatives --set java /usr/lib/jvm/java-21-openjdk-amd64/bin/java 2>/dev/null || true

# ── Verify Java ───────────────────────────────────────────────
java_check = !java -version 2>&1
java_line = next((l for l in java_check if 'version' in l and 'warning' not in l and 'cgroup' not in l.lower()), '')
if '21' in java_line:
    print(f"✅ Java 21 active: {java_line}")
elif java_line:
    print(f"⚠️  Unexpected Java version: {java_line}")
else:
    print("✅ Java 21 installed (cgroup warnings above are harmless — Colab quirk, ignore them)")

# ── Jar name mapping ─────────────────────────────────────────
jar_map = {
    'paper':   'server.jar',
    'fabric':  'fabric-server-launch.jar',
    'forge':   'forge.jar',
    'generic': 'server.jar'
}
jar_name = jar_map.get(server_type, 'server.jar')

# ── Optimized JVM Flags ───────────────────────────────────────
if server_type == 'paper':
    JVM_FLAGS = (
        "-XX:+UseG1GC "
        "-XX:+ParallelRefProcEnabled "
        "-XX:MaxGCPauseMillis=200 "
        "-XX:+UnlockExperimentalVMOptions "
        "-XX:+DisableExplicitGC "
        "-XX:+AlwaysPreTouch "
        "-XX:G1NewSizePercent=30 "
        "-XX:G1MaxNewSizePercent=40 "
        "-XX:G1HeapRegionSize=8M "
        "-XX:G1ReservePercent=20 "
        "-XX:G1HeapWastePercent=5 "
        "-XX:G1MixedGCCountTarget=4 "
        "-XX:InitiatingHeapOccupancyPercent=15 "
        "-XX:G1MixedGCLiveThresholdPercent=90 "
        "-XX:G1RSetUpdatingPauseTimePercent=5 "
        "-XX:SurvivorRatio=32 "
        "-XX:+PerfDisableSharedMem "
        "-XX:MaxTenuringThreshold=1 "
        "-XX:+UseStringDeduplication "
        "-XX:+OptimizeStringConcat "
        "-Dusing.aikars.flags=https://mcflags.emc.gs "
        "-Daikars.new.flags=true"
    )
else:
    JVM_FLAGS = "-XX:+UseG1GC -XX:+ParallelRefProcEnabled -XX:MaxGCPauseMillis=200"

MEMORY_FLAGS = f"-Xms{MEMORY} -Xmx{MEMORY}"

# ── Install playit.gg tunnel ──────────────────────────────────
print("\n🌐 Installing playit.gg tunnel...")
!curl -SsL https://playit-cloud.github.io/ppa/key.gpg | gpg --dearmor | sudo tee /etc/apt/trusted.gpg.d/playit.gpg >/dev/null
!echo 'deb [signed-by=/etc/apt/trusted.gpg.d/playit.gpg] https://playit-cloud.github.io/ppa/data ./' | sudo tee /etc/apt/sources.list.d/playit-cloud.list >/dev/null
!sudo apt-get update -qq && sudo apt-get install -y -qq playit
print("✅ playit.gg installed.")

# ── Start playit.gg via PTY — captures claim URL correctly ──
#
# ROOT CAUSE: playit detects it's not connected to a real terminal
# (TTY) when stdout is a pipe, so it suppresses all output entirely.
# FIX: Launch playit inside a pty (pseudo-terminal) so it thinks
# it's talking to a real terminal and outputs the claim URL.
#
import pty, os, select, re

print("\n🔗 Starting playit.gg tunnel...")
print("   → Waiting for claim URL (up to 90s)...")
print("   → After claiming, your server IP will be in the playit dashboard.\n")

# Open a PTY pair — master side we read from, slave side playit writes to
master_fd, slave_fd = pty.openpty()

playit_proc = subprocess.Popen(
    ['playit'],
    stdin=slave_fd,
    stdout=slave_fd,
    stderr=slave_fd,
    close_fds=True
)
os.close(slave_fd)  # parent doesn't need slave end

claim_url = None
deadline  = time.time() + 90
buf       = ''

while time.time() < deadline:
    try:
        ready, _, _ = select.select([master_fd], [], [], 1.0)
        if not ready:
            continue
        chunk = os.read(master_fd, 4096).decode('utf-8', errors='replace')
        buf += chunk
        # Process complete lines
        while '\n' in buf:
            line, buf = buf.split('\n', 1)
            # Strip ANSI escape codes
            clean = re.sub(r'\x1b\[[0-9;]*[mABCDEFGHJKLMnsu]', '', line).strip()
            if not clean:
                continue
            if any(kw in clean.lower() for kw in ['https://', 'claim', 'tunnel', 'address', 'error', 'connect']):
                print(f"\n🔗 [playit] {clean}", flush=True)
            else:
                print(f"   [playit] {clean}", flush=True)
            if 'https://' in clean and 'claim' in clean.lower():
                claim_url = re.search(r'https://\S+', clean)
                claim_url = claim_url.group(0) if claim_url else clean
                break
        if claim_url:
            break
    except OSError:
        print("⚠️  playit process closed.")
        break

print()
if claim_url:
    print("┌─────────────────────────────────────────────────────────┐")
    print("│  ✅  CLAIM URL — open this ONCE in your browser:        │")
    print(f"│  {claim_url:<55}  │")
    print("└─────────────────────────────────────────────────────────┘")
    print()
else:
    print("⚠️  Claim URL not detected — check [playit] lines above.")
    print("   → If already claimed before, tunnel will connect automatically.\n")

# Drain remaining PTY output in background
def _drain_pty(fd):
    try:
        while True:
            r, _, _ = select.select([fd], [], [], 2.0)
            if not r:
                continue
            data = os.read(fd, 4096).decode('utf-8', errors='replace')
            for ln in data.split('\n'):
                c = re.sub(r'\x1b\[[0-9;]*[mABCDEFGHJKLMnsu]', '', ln).strip()
                if c:
                    print(f"   [playit] {c}", flush=True)
    except OSError:
        pass

threading.Thread(target=_drain_pty, args=(master_fd,), daemon=True).start()

# ── Start Minecraft Server ────────────────────────────────────
print(f"🟢 Starting Minecraft server ({server_type} {server_version})...")
print(f"   JAR: {jar_name}")
print(f"   RAM: {MEMORY} allocated")
print("   Waiting for server to finish loading...\n")

!java {MEMORY_FLAGS} {JVM_FLAGS} -jar {jar_name} nogui

---
# ❓ FAQ & Tips

| Question | Answer |
|---|---|
| **Does the IP change every session?** | Yes. Check playit.gg dashboard for your permanent tunnel address after claiming. |
| **How long does the server stay up?** | Google Colab runs for ~12 hours per session. Keep the tab active to avoid disconnection. |
| **How much RAM does Colab give?** | ~12–13 GB RAM, dual-core ~2.2 GHz CPU. We allocate 10G to the server. |
| **Can I install plugins?** | Yes! Drop `.jar` plugins into the `/plugins` folder in your Drive server folder. |
| **Can I use mods?** | Yes — choose `fabric` or `forge` as `SERVER_TYPE` in Step 1. |
| **Server crashes on start?** | Make sure you ran Step 1 first. Also verify the `.jar` file exists in your Drive folder. |
| **Why Java 21?** | Minecraft 1.21.x requires Java 21+. It also brings major performance improvements. |
| **Claim URL not showing?** | Check the `[playit]` lines above. If already claimed previously, tunnel auto-connects. |

---
### ⚡ Performance Tips
- Use **Paper** — it's significantly faster than vanilla for multiplayer.
- Install **Chunky** plugin to pre-generate chunks before players join.
- Keep view-distance at **8–10** in `server.properties` for better TPS.
- Add **spark** plugin to profile lag if TPS drops.

---
*Made by **FTGAMERV2** | v2.0 | 2026*